# Chapter 02 — Advanced Transforms, Side Inputs & BigQuery I/O

Welcome to **Chapter 02** of the **Google Cloud Dataflow Workshop**!

In this chapter, you will learn production-grade Apache Beam patterns for building resilient, enterprise data pipelines:
1. **`DoFn` Lifecycle Methods**: Properly managing worker setup and teardown.
2. **Side Inputs (`beam.pvalue.AsDict`)**: Broadcasting reference lookup data without expensive network shuffles.
3. **Tagged Multi-Outputs & Dead-Letter Queues (DLQ)**: Splitting streams into clean records vs. poison records using `beam.pvalue.TaggedOutput`.
4. **Google Cloud BigQuery Storage Write API**: Efficient, exactly-once ingestion into BigQuery tables.

## 1. Environment Setup & Kernel Selection

Make sure your notebook is running inside the isolated **`.venv`** virtual environment managed by **`uv`**:

1. **Terminal Activation (optional)**: `source .venv/bin/activate`
2. **Jupyter Kernel**: Click the **Kernel Picker** in the top-right corner of this editor and select **`Python (dataflow-workshops)`** (or browse to `.venv/bin/python`).

Run the verification cell below to confirm your setup:

In [ ]:
import os
import sys

# Verify Python executable location
print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

## 2. Inspecting the Raw Data & Reference Dimensions

In this chapter, we work with two datasets located in `data/`:
1. **`data/sales_data_with_errors.csv`**: Raw retail transactions containing valid records and intentional corruptions (negative amounts, missing product IDs, malformed dates).
2. **`data/product_catalog.json`**: A product dimension catalog containing metadata (`product_name`, `category`, `unit_cost`).

Let's inspect both files:

In [ ]:
from pathlib import Path

print("=== [1] Raw Sales Transactions (with errors) ===")
print(Path("data/sales_data_with_errors.csv").read_text())

print("=== [2] Product Dimension Catalog ===")
print(Path("data/product_catalog.json").read_text())

## 3. Understanding the `DoFn` Lifecycle

A `beam.DoFn` is executed across distributed worker processes and threads. Apache Beam provides explicit lifecycle hooks:

| Method | Invocation Frequency | Best Used For |
| :--- | :--- | :--- |
| **`setup(self)`** | Once per worker thread when initializing | Compiling regexes, opening database connection pools, loading ML models |
| **`start_bundle(self)`** | Once before each batch of elements | Initializing in-memory batch accumulators |
| **`process(self, element)`** | Once per incoming element | Main transformation logic, yielding elements |
| **`finish_bundle(self)`** | Once after each batch of elements | Flushing buffered writes |
| **`teardown(self)`** | Once when worker thread shuts down | Closing network connections, cleaning up disk cache |

> **Performance Rule**: NEVER compile regexes or establish database connections inside `process()`! Doing so creates massive overhead for every element. Always use `setup()`!

## 4. Side Inputs (`beam.pvalue.AsDict`)

How do we join sales transactions with product metadata?
* In a full distributed join (`CoGroupByKey`), Dataflow shuffles both datasets across all workers—very costly if one dataset is small.
* With a **Side Input**, Beam broadcasts the small dimension table to every worker VM as an in-memory dictionary (`beam.pvalue.AsDict`).
* The worker accesses the lookup table with instant $O(1)$ dictionary lookups:

```python
class EnrichFn(beam.DoFn):
    def process(self, transaction, catalog):
        # 'catalog' is the Side Input dictionary injected into every process call!
        prod_info = catalog.get(transaction['product_id'])
        ...
```

Let's write `ParseCatalogFn` to parse the catalog into `(product_id, dict)` key-value pairs:

In [ ]:
import json

class ParseCatalogFn(beam.DoFn):
    """Parses JSON catalog lines into (product_id, metadata) pairs for Side Input."""
    def process(self, line: str):
        line = line.strip()
        if not line or line in ("[", "]"):
            return
        clean_line = line.rstrip(",")
        try:
            item = json.loads(clean_line)
            if "product_id" in item:
                yield (item["product_id"], item)
        except json.JSONDecodeError:
            pass

## 5. Branching with Tagged Multi-Outputs & Dead-Letter Queues (DLQ)

In production, never let corrupt records crash your pipeline or drop silently without a trace.

Using **`beam.pvalue.TaggedOutput`**, we route records into two independent streams:
1. **Main Output (`valid_sales`)**: Clean, validated records enriched with catalog metadata.
2. **Tagged Output (`dead_letter`)**: Poison records with their original raw payload, exact error reason, and timestamp.

Let's inspect our `ValidateAndEnrichFn` implementation:

In [ ]:
import datetime
import re

DEAD_LETTER_TAG = "dead_letter"

class ValidateAndEnrichFn(beam.DoFn):
    def setup(self):
        # Compiled once per worker instance
        self.date_regex = re.compile(r"^\d{4}-\d{2}-\d{2}$")
        
    def process(self, line: str, catalog: dict):
        line = line.strip()
        if not line or line.startswith("transaction_id"):
            return
            
        parts = [p.strip() for p in line.split(",")]
        now_ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
        
        # Rule 1: Field count
        if len(parts) != 4:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Expected 4 fields, got {len(parts)}",
                "timestamp": now_ts
            })
            return
            
        tx_id, prod_id, amount_str, tx_date = parts
        
        # Rule 2: Missing product ID
        if not prod_id:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": "Missing product_id",
                "timestamp": now_ts
            })
            return
            
        # Rule 3: Valid positive amount
        try:
            amount = float(amount_str)
            if amount <= 0:
                yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                    "raw_line": line,
                    "error_reason": f"Non-positive amount: {amount}",
                    "timestamp": now_ts
                })
                return
        except ValueError:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Invalid numeric amount '{amount_str}'",
                "timestamp": now_ts
            })
            return
            
        # Rule 4: Valid date format
        if not self.date_regex.match(tx_date):
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Invalid date format '{tx_date}' (must be YYYY-MM-DD)",
                "timestamp": now_ts
            })
            return
            
        # Rule 5: Side Input lookup in catalog
        prod_info = catalog.get(prod_id)
        if not prod_info:
            yield beam.pvalue.TaggedOutput(DEAD_LETTER_TAG, {
                "raw_line": line,
                "error_reason": f"Product '{prod_id}' not found in catalog",
                "timestamp": now_ts
            })
            return
            
        # Successful enrichment emitted to default (main) stream
        unit_cost = prod_info.get("unit_cost", 0.0)
        profit = round(amount - unit_cost, 2)
        
        yield {
            "transaction_id": tx_id,
            "product_id": prod_id,
            "product_name": prod_info.get("product_name"),
            "category": prod_info.get("category"),
            "amount": amount,
            "profit": profit,
            "transaction_date": tx_date,
            "processed_at": now_ts,
        }

## 6. Pipeline Architecture (DAG)

Notice how the **`PCollection`** (cylinders) and **`PTransform`** (rectangles) branch into two destinations:

![Chapter 02 Pipeline Architecture](../artifacts/02_side_inputs_and_bq_dag.png)

<details>
<summary><b>Click to expand raw Mermaid flowchart definition</b></summary>

```mermaid
flowchart TD
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef dlq fill:#fce8e6,stroke:#ea4335,stroke-width:2px,color:#c5221f,font-weight:bold;

    subgraph Inputs["1. Raw Ingestion"]
        SALES_CSV[("[( sales_data_with_errors.csv )]<br/>Raw Transactions (with errors)")]:::pcoll
        CATALOG_JSON[("[( product_catalog.json )]<br/>Reference Product Dimension")]:::pcoll
    end

    subgraph SideInput["2. Side Input Creation"]
        T_CAT_READ["PTransform: beam.io.ReadFromText<br/>Name: 'ReadCatalog'"]:::ptrans
        T_CAT_PARSE["PTransform: beam.ParDo(ParseCatalogFn)<br/>Name: 'ParseCatalog'"]:::ptrans
        PC_CATALOG[("[( PCollection&lt;tuple&gt; )]<br/>(product_id, metadata)")]:::pcoll
    end

    subgraph Core["3. Processing & Validation Engine"]
        T_SALES_READ["PTransform: beam.io.ReadFromText<br/>Name: 'ReadSales'"]:::ptrans
        PC_RAW_SALES[("[( PCollection&lt;str&gt; )]<br/>Raw CSV Lines")]:::pcoll
        T_ENRICH["PTransform: beam.ParDo(ValidateAndEnrichFn)<br/>Name: 'ValidateAndEnrich'"]:::ptrans
    end

    subgraph Outputs["4. Dual-Sink Routing"]
        PC_VALID[("[( PCollection&lt;dict&gt; (Main Output) )]<br/>Enriched Valid Sales")]:::pcoll
        PC_DLQ[("[( PCollection&lt;dict&gt; ('dead_letter' Tag) )]<br/>Poison Records")]:::dlq
        
        T_BQ["PTransform: beam.io.WriteToBigQuery<br/>Name: 'WriteToBigQuery' (Storage Write API)"]:::ptrans
        BQ_TABLE[("[( BigQuery Table )]<br/>curated_sales")]:::sink
        
        T_DLQ["PTransform: beam.io.WriteToText<br/>Name: 'WriteDLQ'"]:::ptrans
        DLQ_FILES[("[( Cloud Storage DLQ )]<br/>dead_letter_records-*.json")]:::dlq
    end

    CATALOG_JSON --> T_CAT_READ
    T_CAT_READ --> T_CAT_PARSE
    T_CAT_PARSE --> PC_CATALOG
    
    SALES_CSV --> T_SALES_READ
    T_SALES_READ --> PC_RAW_SALES
    PC_RAW_SALES --> T_ENRICH

    PC_CATALOG -. "Side Input: beam.pvalue.AsDict" .-> T_ENRICH

    T_ENRICH -- "Main Output" --> PC_VALID
    T_ENRICH -- "Tagged Output ('dead_letter')" --> PC_DLQ

    PC_VALID --> T_BQ
    T_BQ --> BQ_TABLE

    PC_DLQ --> T_DLQ
    T_DLQ --> DLQ_FILES
```
</details>

## 7. Executing the Pipeline Locally with `DirectRunner`

Let's run the complete pipeline locally and write outputs to `output/`:

In [ ]:
import shutil

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

valid_prefix = "output/notebook_valid_sales"
dlq_prefix = "output/notebook_dlq_records"

with beam.Pipeline() as p:
    # 1. Build Side Input Collection
    catalog_pcoll = (
        p
        | "ReadCatalog" >> beam.io.ReadFromText("data/product_catalog.json")
        | "ParseCatalog" >> beam.ParDo(ParseCatalogFn())
    )
    
    # 2. Validate, Enrich, and Branch
    results = (
        p
        | "ReadSales" >> beam.io.ReadFromText("data/sales_data_with_errors.csv")
        | "ValidateAndEnrich" >> beam.ParDo(
            ValidateAndEnrichFn(),
            catalog=beam.pvalue.AsDict(catalog_pcoll)
        ).with_outputs(DEAD_LETTER_TAG, main="valid_sales")
    )
    
    valid_sales = results.valid_sales
    dead_letter = results[DEAD_LETTER_TAG]
    
    # 3. Write Valid Records Locally
    (
        valid_sales
        | "FormatValid" >> beam.Map(json.dumps)
        | "WriteValid" >> beam.io.WriteToText(valid_prefix, file_name_suffix=".json")
    )
    
    # 4. Write Poison Records to Dead-Letter Queue Locally
    (
        dead_letter
        | "FormatDLQ" >> beam.Map(json.dumps)
        | "WriteDLQ" >> beam.io.WriteToText(dlq_prefix, file_name_suffix=".json")
    )

print("✅ Pipeline completed locally!")

Let's inspect the results:

In [ ]:
print("=== [1] Valid Enriched Sales Records ===")
for f in Path("output").glob("notebook_valid_sales*.json"):
    print(f.read_text())

print("=== [2] Dead-Letter Poison Records ===")
for f in Path("output").glob("notebook_dlq_records*.json"):
    print(f.read_text())

## 8. Loading into Google Cloud BigQuery

In production on Google Cloud, we swap the local JSON sink with **`beam.io.WriteToBigQuery`** using the **BigQuery Storage Write API**:

```python
CURATED_SALES_SCHEMA = {
    "fields": [
        {"name": "transaction_id", "type": "STRING", "mode": "REQUIRED"},
        {"name": "product_id", "type": "STRING", "mode": "REQUIRED"},
        {"name": "product_name", "type": "STRING", "mode": "NULLABLE"},
        {"name": "category", "type": "STRING", "mode": "NULLABLE"},
        {"name": "amount", "type": "FLOAT64", "mode": "REQUIRED"},
        {"name": "profit", "type": "FLOAT64", "mode": "NULLABLE"},
        {"name": "transaction_date", "type": "DATE", "mode": "REQUIRED"},
        {"name": "processed_at", "type": "TIMESTAMP", "mode": "REQUIRED"},
    ]
}

valid_sales | "WriteToBigQuery" >> beam.io.WriteToBigQuery(
    table="YOUR_PROJECT:dataflow_workshop.curated_sales",
    schema=CURATED_SALES_SCHEMA,
    write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
    create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
    method=beam.io.WriteToBigQuery.Method.STORAGE_WRITE_API,
)
```

### Deploying to Google Cloud Dataflow
To deploy this pipeline to Google Cloud, execute the automated cloud submission script:

```bash
bash 02_transforms_and_bigquery/scripts/02_run_dataflow.sh
```

---

## Summary & What's Next

In this chapter, you mastered:
* **`DoFn` lifecycle methods** (`setup`, `process`, `teardown`) to properly manage worker resources.
* **Side Inputs (`beam.pvalue.AsDict`)** to perform $O(1)$ dimension table lookups without expensive distributed shuffles.
* **Tagged Multi-Outputs (`beam.pvalue.TaggedOutput`)** to isolate corrupt records into a Dead-Letter Queue without crashing the pipeline.
* **BigQuery Storage Write API** for scalable, cost-effective data warehousing.

👉 **Next: [Chapter 03 — Real-Time Streaming & Windowing](../03_realtime_streaming/README.md)**:
We transition from batch files to continuous unbounded streaming from **Google Cloud Pub/Sub**, exploring **Fixed/Sliding Windows**, **Watermarks**, and **Late-Data Handling**!